# S05 Lab — Northstar Deployment and Context Bake-Off

## Which weights, where they run, and what you send them

This lab holds the technique constant — a single structured call, S02's contract — and varies three things one at a time: **which model** (closed-weight vs. open-weight), **where it runs** (hosted vs. local), and **how much context it gets** (full knowledge base vs. trimmed to the relevant section). A real tokenizer measures every cost and savings claim; nothing here is estimated by eye.

## Parts:

PART 1: The knowledge base and ticket set

PART A: Model/deployment comparison (context held constant)

PART B: Context comparison (configuration held constant)

PART C: FDE decision exercise

PART D: Optional live mode

> [!NOTE]
> No API key, network access, or paid account is required for Parts 1, A, and B — the tokenizer runs fully offline. The optional live section (Part D) is off by default (`RUN_LIVE = False`). All tickets and knowledge-base text below are synthetic.

In [ ]:
# %pip install pandas 'pydantic>=2,<3' tiktoken

from typing import Literal

import pandas as pd
import tiktoken
from pydantic import BaseModel, Field

ENCODING = tiktoken.get_encoding("o200k_base")


def count_tokens(text):
    """The real token count for a piece of text, using a standalone tokenizer."""
    return len(ENCODING.encode(text))

## Part 1: The knowledge base and ticket set

The same decision contract from S02, reused unchanged.

In [ ]:
class TicketDecision(BaseModel):
    """The same decision contract validated in S02."""

    category: Literal["billing", "technical", "access", "general"]
    priority: Literal["low", "medium", "high"]
    needs_human: bool
    summary: str = Field(min_length=1, max_length=240)

Northstar's support knowledge base: a general-rules section that always applies, plus one section per ticket category. The **access** section's "repeated 401 errors" rule only applies after an SSO change or certificate rotation — a detail Part B's mini exercise comes back to.

In [ ]:
GENERAL_RULES = """# Northstar support knowledge base — general rules

Escalate to a human (needs_human = true) whenever a refund, payout, or compliance action is involved, when a customer reports an outage affecting more than one user, or when the correct category is still ambiguous after reading the section below. Priority is high when a core workflow is blocked right now, medium when there is a confirmed problem with no immediate blocker, and low otherwise."""

SECTIONS = {
    "billing": """## Billing

Duplicate charges, incorrect add-on fees, proration questions, and invoice or receipt requests are billing issues. A billing dispute that requires comparing two different internal data sources — for example, a usage dashboard figure against raw usage logs — must be escalated to a human; never resolve a cross-source discrepancy from the ticket text alone, since the ticket text cannot tell you which source is actually correct.""",
    "technical": """## Technical

API errors, webhook failures, failed or empty scheduled jobs, and measurable performance regressions are technical issues. A technical issue is high priority when it blocks a core workflow for multiple users right now, medium when there is a confirmed regression with a workaround available, and low for a cosmetic or isolated defect.""",
    "access": """## Access

Login failures, SSO configuration, invite delivery, and offboarding or access-revocation requests are access issues. Any report of repeated 401 errors that follows an SSO configuration change or a certificate rotation must be treated as access, priority high, and flagged for human review — that specific pattern has caused full lockouts before.""",
    "general": """## General

Documentation questions, plan-comparison questions, and status-page or subscription requests that report no actual problem are general issues, priority low, with no human review needed.""",
}

FULL_SECTIONS = {"billing", "technical", "access", "general"}


def full_context():
    """The general rules plus every category section."""
    return "\n\n".join([GENERAL_RULES] + [SECTIONS[c] for c in ["billing", "technical", "access", "general"]])


def trimmed_context(topic_hint):
    """The general rules plus only the section matching the ticket's (free, already-known) topic hint."""
    return "\n\n".join([GENERAL_RULES, SECTIONS[topic_hint]])

In [ ]:
TICKETS = [
    {"ticket_id": "T-301", "topic_hint": "billing",
     "text": "Our usage dashboard shows we're at 140% of our plan's API limit this month, but our own server logs only show 95%. Which number should we trust for this month's invoice?",
     "gold": {"category": "billing", "priority": "medium", "needs_human": True}},
    {"ticket_id": "T-302", "topic_hint": "technical",
     "text": "Our mobile app can't connect to the API — I keep getting 401 unauthorized errors since this morning's deploy.",
     "gold": {"category": "technical", "priority": "medium", "needs_human": False}},
    {"ticket_id": "T-303", "topic_hint": "billing",
     "text": "There's a $15 add-on fee on my invoice that I never signed up for.",
     "gold": {"category": "billing", "priority": "medium", "needs_human": True}},
    {"ticket_id": "T-304", "topic_hint": "billing",
     "text": "Could I get PDF copies of our last three invoices for our finance team?",
     "gold": {"category": "billing", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-305", "topic_hint": "technical",
     "text": "Our nightly data export job has come back empty two nights in a row.",
     "gold": {"category": "technical", "priority": "medium", "needs_human": False}},
    {"ticket_id": "T-306", "topic_hint": "technical",
     "text": "API response times have roughly doubled for everyone on our team since yesterday's release.",
     "gold": {"category": "technical", "priority": "high", "needs_human": True}},
    {"ticket_id": "T-307", "topic_hint": "access",
     "text": "Our newest hire never received the invite email to set up their account.",
     "gold": {"category": "access", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-308", "topic_hint": "access",
     "text": "Please revoke system access immediately for an employee who left the company this morning.",
     "gold": {"category": "access", "priority": "high", "needs_human": True}},
    {"ticket_id": "T-309", "topic_hint": "general",
     "text": "Is there a status page we can subscribe to for incident updates?",
     "gold": {"category": "general", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-310", "topic_hint": "general",
     "text": "What's the difference between the Growth and Scale plans?",
     "gold": {"category": "general", "priority": "low", "needs_human": False}},
]

pd.DataFrame(TICKETS)[["ticket_id", "topic_hint", "text"]]

A deterministic mock decision function with two **designed** exceptions: a capability-limited error on the one hard, cross-referencing ticket (any model that isn't `"closed"`), and a context-dilution error on the 401 ticket (any open-weight model, only when the access section happens to be present in context).

In [ ]:
def decide(ticket, sections_included, capability):
    """Mock decision: correct except for two designed, explainable exceptions."""
    gold = ticket["gold"]

    if ticket["ticket_id"] == "T-301" and capability != "closed":
        # A less capable model defaults to reading this as a dashboard display bug,
        # instead of recognizing a billing dispute that needs human reconciliation.
        return {"category": "technical", "priority": "low", "needs_human": False}

    if ticket["ticket_id"] == "T-302" and "access" in sections_included and capability != "closed":
        # Context dilution: the access section's unrelated "401" rule distracts a
        # less capable model, but only when that section is actually in context.
        return {"category": "access", "priority": "high", "needs_human": True}

    return dict(gold)


def is_right(result, gold):
    """True when category, priority, and needs_human all match the reference label."""
    return result["category"] == gold["category"] and result["priority"] == gold["priority"] and result["needs_human"] == gold["needs_human"]

Three configurations: the same underlying classification task, run by a closed-weight hosted model, an open-weight hosted model, and the same open-weight model run locally. Cost and latency rates below are this lab's own illustrative stand-ins — only the **token counts** they're multiplied by are real.

In [ ]:
CONFIGS = {
    "closed_hosted": {"capability": "closed", "cost_per_1k": 5.0, "seconds_per_1k": 1.5, "memory_gb": None, "self_hostable": False},
    "open_hosted": {"capability": "open", "cost_per_1k": 2.0, "seconds_per_1k": 2.0, "memory_gb": None, "self_hostable": True},
    "open_local": {"capability": "open", "cost_per_1k": 0.0, "seconds_per_1k": 9.0, "memory_gb": 6.0, "self_hostable": True},
}


def run_config(config_name, tickets, context_mode):
    """Run one configuration over all tickets with a fixed context mode ('full' or 'trimmed')."""
    cfg = CONFIGS[config_name]
    rows = []
    for t in tickets:
        if context_mode == "full":
            ctx, sections_included = full_context(), FULL_SECTIONS
        else:
            ctx, sections_included = trimmed_context(t["topic_hint"]), {"general_rules", t["topic_hint"]}
        tokens = count_tokens(ctx + "\n\n" + t["text"])
        result = decide(t, sections_included, cfg["capability"])
        rows.append({
            "ticket_id": t["ticket_id"],
            "config": config_name,
            "context_mode": context_mode,
            "tokens": tokens,
            "cost": round(tokens / 1000 * cfg["cost_per_1k"], 3),
            "seconds": round(tokens / 1000 * cfg["seconds_per_1k"], 2),
            **result,
            "right": is_right(result, t["gold"]),
        })
    return pd.DataFrame(rows)


def summarize(df, label):
    """Print and return accuracy, average tokens, total illustrative cost, and total illustrative seconds for one run."""
    right = int(df["right"].sum())
    total = len(df)
    avg_tokens = df["tokens"].mean()
    cost = df["cost"].sum()
    seconds = df["seconds"].sum()
    print(f"{label}: {right} of {total} right, avg {avg_tokens:.1f} tokens/ticket, {cost:.2f} cost-units total, {seconds:.1f} seconds total")
    return {"label": label, "right": right, "total": total, "accuracy": right / total, "avg_tokens": avg_tokens, "cost": cost, "seconds": seconds}

## Part A: Model/deployment comparison (context held constant)

All three configurations, full context, every time.

In [ ]:
closed_hosted_full = run_config("closed_hosted", TICKETS, "full")
open_hosted_full = run_config("open_hosted", TICKETS, "full")
open_local_full = run_config("open_local", TICKETS, "full")

summary_closed = summarize(closed_hosted_full, "Closed, hosted")
summary_open_hosted = summarize(open_hosted_full, "Open-weight, hosted")
summary_open_local = summarize(open_local_full, "Open-weight, local")

part_a = pd.DataFrame([summary_closed, summary_open_hosted, summary_open_local])
part_a["memory_gb"] = [CONFIGS[c]["memory_gb"] for c in ["closed_hosted", "open_hosted", "open_local"]]
part_a["self_hostable"] = [CONFIGS[c]["self_hostable"] for c in ["closed_hosted", "open_hosted", "open_local"]]
part_a

In [ ]:
open_hosted_full[open_hosted_full["right"] == False][["ticket_id", "category", "priority", "needs_human", "right"]]

The two open-weight rows should score identically — same weights, same missed tickets. If they don't, that's a bug to find (handout §2), not a deployment-location effect to explain away.

**Commercial implications:** The closed model's accuracy edge comes entirely from one cross-referencing ticket. That's a real, measured capability gap — but a single ticket on a 10-ticket trial can't tell you how often real traffic looks like that case, which is exactly what a production recommendation would need before paying the closed model's cost premium on every ticket.

## Part B: Context comparison (configuration held constant)

The open-weight, hosted configuration, run once with full context and once with trimmed context.

In [ ]:
open_hosted_trimmed = run_config("open_hosted", TICKETS, "trimmed")

summary_full = summarize(open_hosted_full, "Open-weight, hosted — full context")
summary_trimmed = summarize(open_hosted_trimmed, "Open-weight, hosted — trimmed context")

part_b = pd.DataFrame([summary_full, summary_trimmed])
part_b["token_reduction"] = 1 - part_b["avg_tokens"] / summary_full["avg_tokens"]
part_b

In [ ]:
# Which ticket's result changed between full and trimmed context?
merged = open_hosted_full.merge(open_hosted_trimmed, on="ticket_id", suffixes=("_full", "_trimmed"))
merged[merged["right_full"] != merged["right_trimmed"]][["ticket_id", "category_full", "category_trimmed", "right_full", "right_trimmed"]]

**Mini exercise:** Before reading the handout's §5, explain in your own words *why* trimming fixed this specific ticket rather than just making the prompt shorter. Check your explanation against the handout once you've written it down.

**Learner modification:** Add a new section to `SECTIONS` (e.g., `"security"`) containing its own trigger phrase, and a new ticket in `TICKETS` whose text contains that phrase but whose `topic_hint` is a *different* category. Run it through `run_config` with `"full"` and `"trimmed"` context and confirm you've reproduced the same kind of context-dilution bug on your own new section.

## Part C: FDE decision exercise

Write the constraint-first recommendation for the data-residency scenario: a prospect requires ticket text never leave their infrastructure, their trial used the closed, hosted configuration, and the account team wants to know if the deal still works with "the good model."

Full exercise, rubric, and worked examples: [`docs/fde-exercises/open-weight-local-deployment-decision.md`](../../../docs/fde-exercises/open-weight-local-deployment-decision.md)

## Part D: Optional live model mode (requires credentials and provider validation)

This section sends the **real** full and trimmed prompts to a real model and compares real token counts and timing against the mock numbers above. It does not run a real local deployment — that's out of scope here; the mock `open_local` configuration's numbers stand in for it.

> [!CAUTION]
> Only ever send synthetic or illustrative data to a live model, never real customer data. This path has not been executed as part of building this lab — review the provider's current structured-output support, data-retention terms, and pricing before running it live.

In [ ]:
# %pip install openai python-dotenv

import os
import time

RUN_LIVE = False
LIVE_MODEL = "openai/gpt-4o-mini"  # replace with a verified, current OpenRouter model ID

if RUN_LIVE:
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(override=True)
    openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
    if not openrouter_api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY in your environment before running the live section.")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

    def decide_live(ticket, context_text, model):
        """A real single structured call, given either the full or trimmed context."""
        start = time.time()
        response = client.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": context_text},
                {"role": "user", "content": ticket["text"]},
            ],
            response_format=TicketDecision,
        )
        return response.choices[0].message.parsed, response.usage.prompt_tokens, round(time.time() - start, 1)

    for ticket in TICKETS:
        for mode, ctx in [("full", full_context()), ("trimmed", trimmed_context(ticket["topic_hint"]))]:
            parsed, tokens, seconds = decide_live(ticket, ctx, LIVE_MODEL)
            print(ticket["ticket_id"], mode, tokens, "tokens", seconds, "s", parsed)
else:
    print("RUN_LIVE is False — skipping the live section. Set RUN_LIVE = True and OPENROUTER_API_KEY to run it.")

---

## About the mentor

**Sanjay Negi** brings 23+ years of industry experience and helps experienced software engineers and technology leaders transition into high-value AI roles, including **AI Engineer**, **Forward Deployed Engineer**, and **GenAI Architect**. He has trained more than 1,100 experienced engineers.

#SkillUpWithSanjay · [Website](https://www.sanjaynegi.in) · [LinkedIn](https://www.linkedin.com/in/sanjay-negi-ai/)